# 🏷️ Hands-On Lab: Image Classification Labeling with Amazon SageMaker Ground Truth

---

## 📋 Lab Overview

**Scenario:** As an AI/ML Engineer at K21 Academy, your role involves preparing high-quality datasets for machine learning models. In this lab, you will set up an **Image Classification Labeling Job** using Amazon SageMaker Ground Truth — entirely through Python code in this notebook.

**What You Will Learn:**
- Create and configure an Amazon S3 bucket programmatically
- Download and upload images to S3 using `boto3`
- Create a **Private Workforce** in SageMaker Ground Truth
- Generate a Ground Truth **manifest file** for your input images
- Create an **Image Classification Labeling Job** via the SageMaker API
- Monitor job progress and retrieve labeled output
- Clean up all resources at the end

**Region:** `us-east-1` (N. Virginia) — **Do not change this.**

**Estimated Cost:** \$0.22 – \$0.50 (within Free Tier limits for first 12 months)

---

> ⚠️ **Prerequisites:**
> - You must have an AWS account with appropriate permissions
> - This notebook should be run inside **Amazon SageMaker Studio** or a **SageMaker Notebook Instance**
> - The IAM role attached to this notebook must have permissions for: `s3`, `sagemaker`, `iam`, `cognito-idp`, `cognito-identity`

---
## 📦 Cell 1: Install & Import Required Libraries

We start by importing all Python libraries needed throughout this lab.
- `boto3` — AWS SDK for Python (to interact with S3, SageMaker, etc.)
- `sagemaker` — High-level SageMaker SDK
- `json`, `time`, `urllib` — Standard Python utilities

In [1]:
# Install/upgrade required libraries (run once)
!pip install --quiet --upgrade boto3 sagemaker requests

import boto3
import sagemaker
import json
import time
import os
import urllib.request
from datetime import datetime
from botocore.exceptions import ClientError

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---
## ⚙️ Cell 2: Configuration — Set Your Lab Variables

Define all the names and settings used throughout this lab.

> 📝 **Note:** S3 bucket names must be globally unique. If `k21academyturk` is taken, add your AWS account ID suffix (handled automatically below).

In [2]:
# ── AWS Session Setup ──────────────────────────────────────────────────────────
session    = boto3.Session(region_name="us-east-1")
account_id = boto3.client("sts").get_caller_identity()["Account"]
region     = "us-east-1"

# ── SageMaker Role ─────────────────────────────────────────────────────────────
# This fetches the IAM role already attached to your SageMaker notebook.
# It needs permissions for S3, SageMaker Ground Truth, and Cognito.
try:
    role = sagemaker.get_execution_role()
except Exception:
    # Fallback: manually specify your role ARN if running outside SageMaker
    role = f"arn:aws:iam::{account_id}:role/SageMakerExecutionRole"

# ── Resource Names ─────────────────────────────────────────────────────────────
BUCKET_NAME       = f"k21academyturk-{account_id}"   # Globally unique bucket name
TEAM_NAME         = "K21Team"                          # Private workforce team name
JOB_NAME          = f"K21academy-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
ORG_NAME          = "K21 Academy"
LABELER_EMAIL     = "ritik@academy.com"           # ← CHANGE THIS to your email

# ── S3 Paths ───────────────────────────────────────────────────────────────────
S3_INPUT_PREFIX   = "ground-truth/input"
S3_OUTPUT_PREFIX  = "ground-truth/output"
MANIFEST_KEY      = f"{S3_INPUT_PREFIX}/manifest.json"

# ── Image Download URLs (from lab guide) ───────────────────────────────────────
IMAGE_URLS = {
    "Happy_face.png": (
        "https://raw.githubusercontent.com/k21academyuk/AWS-GenAI_AIML/main/Happy_face.png"
    ),
    "Building_with_plane.png": (
        "https://raw.githubusercontent.com/k21academyuk/AWS-GenAI_AIML/main/Buidling_with_plane.png"
    ),
}

# ── Labels (same as the console lab) ──────────────────────────────────────────
LABELS = ["Flight", "Building", "Happy Face"]

print(f"✅ Configuration complete!")
print(f"   Account ID  : {account_id}")
print(f"   Region      : {region}")
print(f"   Bucket Name : {BUCKET_NAME}")
print(f"   Job Name    : {JOB_NAME}")
print(f"   IAM Role    : {role}")
print()
print("⚠️  ACTION REQUIRED: Update LABELER_EMAIL above before continuing!")

✅ Configuration complete!
   Account ID  : 745404749079
   Region      : us-east-1
   Bucket Name : k21academyturk-745404749079
   Job Name    : K21academy-20260612-085352
   IAM Role    : arn:aws:iam::745404749079:role/SageMakerExecutionRole

⚠️  ACTION REQUIRED: Update LABELER_EMAIL above before continuing!


---
## 🪣 Cell 3: Create an S3 Bucket

**Console equivalent:** S3 → Create bucket → Name: `k21academyturk`

We create a private S3 bucket in `us-east-1` to store our images and labeling outputs.

In [3]:
s3_client = session.client("s3")

def create_s3_bucket(bucket_name, region):
    """Create an S3 bucket; skip if it already exists."""
    try:
        if region == "us-east-1":
            # us-east-1 does NOT use LocationConstraint
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"✅ S3 bucket '{bucket_name}' created successfully.")
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
            print(f"ℹ️  Bucket '{bucket_name}' already exists — skipping creation.")
        else:
            raise

create_s3_bucket(BUCKET_NAME, region)

# Block all public access (security best practice)
s3_client.put_public_access_block(
    Bucket=BUCKET_NAME,
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,
        "IgnorePublicAcls": True,
        "BlockPublicPolicy": True,
        "RestrictPublicBuckets": True,
    }
)
print("✅ Public access blocked — bucket is private.")

✅ S3 bucket 'k21academyturk-745404749079' created successfully.
✅ Public access blocked — bucket is private.


---
## 🔒 Cell 3b: Bucket Policy + CORS Configuration

**Two things required for Ground Truth to work with S3 images:**

1. **Bucket Policy** — allows `sagemaker.amazonaws.com` service to read/write S3
2. **CORS Configuration** — Ground Truth labeling portal runs in the browser; the browser blocks image loads from S3 unless CORS headers are present. Without this you get:
   > *`failure-reason: S3 bucket does not have a CORS configuration`*

> ⚠️ Both must be applied **before** creating the labeling job.

In [4]:
# ── Grant SageMaker Ground Truth access to S3 (Bucket Policy) ───────────────
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "GroundTruthServiceAccess",
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*"
            ]
        }
    ]
}
s3_client.put_bucket_policy(
    Bucket=BUCKET_NAME,
    Policy=json.dumps(bucket_policy)
)
print("✅ Bucket policy applied.")

# ── Add CORS configuration ────────────────────────────────────────────────────
# Ground Truth labeling portal runs in the browser. The browser enforces CORS.
# Without this, images cannot load in the labeling UI → FailedNonRetryableError.
# This is the exact fix for: 'S3 bucket does not have a CORS configuration'
cors_config = {
    "CORSRules": [
        {
            "AllowedHeaders": ["*"],
            "AllowedMethods": ["GET", "HEAD", "PUT", "POST"],
            "AllowedOrigins": ["*"],
            "ExposeHeaders": ["ETag", "x-amz-request-id"],
            "MaxAgeSeconds": 3000
        }
    ]
}
s3_client.put_bucket_cors(
    Bucket=BUCKET_NAME,
    CORSConfiguration=cors_config
)
print("✅ CORS configuration applied.")
print(f"   Bucket: {BUCKET_NAME}")
print("   Images will now load correctly in the Ground Truth labeling portal.")

✅ Bucket policy applied.
✅ CORS configuration applied.
   Bucket: k21academyturk-745404749079
   Images will now load correctly in the Ground Truth labeling portal.


---
## 📥 Cell 4: Download Images and Upload to S3

**Console equivalent:** Download images → S3 → Upload

We download the two lab images (Happy Face & Building with Plane) from GitHub and upload them directly to S3 without saving to disk.

In [5]:
import requests

uploaded_s3_uris = []  # We'll collect S3 URIs for the manifest

for filename, url in IMAGE_URLS.items():
    print(f"⬇️  Downloading: {filename} ...")
    response = requests.get(url)
    response.raise_for_status()
    
    # S3 key (path inside the bucket)
    s3_key = f"{S3_INPUT_PREFIX}/{filename}"
    
    # Upload image bytes directly to S3
    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=s3_key,
        Body=response.content,
        ContentType="image/png"
    )
    
    s3_uri = f"s3://{BUCKET_NAME}/{s3_key}"
    uploaded_s3_uris.append(s3_uri)
    print(f"✅ Uploaded → {s3_uri}")

print(f"\n📂 Total images uploaded: {len(uploaded_s3_uris)}")

⬇️  Downloading: Happy_face.png ...


✅ Uploaded → s3://k21academyturk-745404749079/ground-truth/input/Happy_face.png
⬇️  Downloading: Building_with_plane.png ...


✅ Uploaded → s3://k21academyturk-745404749079/ground-truth/input/Building_with_plane.png

📂 Total images uploaded: 2


---
## 📄 Cell 5: Create the Ground Truth Input Manifest File

**Console equivalent:** SageMaker Ground Truth → Automated Data Setup → Browse S3

SageMaker Ground Truth requires an **input manifest** — a JSON Lines file where each line points to one data object (image). We generate this file and upload it to S3.

In [6]:
# Build manifest: one JSON object per line, each referencing an S3 image URI
manifest_lines = []
for s3_uri in uploaded_s3_uris:
    manifest_lines.append(json.dumps({"source-ref": s3_uri}))

manifest_content = "\n".join(manifest_lines)

# Upload manifest to S3
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=MANIFEST_KEY,
    Body=manifest_content.encode("utf-8"),
    ContentType="application/jsonlines"
)

MANIFEST_S3_URI = f"s3://{BUCKET_NAME}/{MANIFEST_KEY}"
OUTPUT_S3_URI   = f"s3://{BUCKET_NAME}/{S3_OUTPUT_PREFIX}/"

print("✅ Manifest file created and uploaded!")
print(f"   Manifest URI : {MANIFEST_S3_URI}")
print(f"   Output URI   : {OUTPUT_S3_URI}")
print()
print("📄 Manifest contents:")
print(manifest_content)

✅ Manifest file created and uploaded!
   Manifest URI : s3://k21academyturk-745404749079/ground-truth/input/manifest.json
   Output URI   : s3://k21academyturk-745404749079/ground-truth/output/

📄 Manifest contents:
{"source-ref": "s3://k21academyturk-745404749079/ground-truth/input/Happy_face.png"}
{"source-ref": "s3://k21academyturk-745404749079/ground-truth/input/Building_with_plane.png"}


---
## 👥 Cell 6: Create a Private Workforce Team

**Console equivalent:** SageMaker → Ground Truth → Labeling Workforces → Private → Create Private Team

A **private workforce** uses a Cognito User Pool to manage labelers. When you create a workteam and invite a member, SageMaker sends an email with login credentials — just like the console flow.

> ⚠️ **Important:** Make sure `LABELER_EMAIL` in Cell 2 is set to **your real email** before running this cell. You will receive an invitation email with your labeling portal credentials.

In [7]:
sm_client = session.client("sagemaker")

# ── Step 1: Check if a private workforce already exists ────────────────────────
# Each AWS account has at most ONE private workforce per region.
workforce_arn = None
user_pool_id  = None
user_pool_client_id = None
cognito_domain = None

try:
    wf_resp = sm_client.describe_workforce(WorkforceName="default")
    workforce_arn = wf_resp["Workforce"]["WorkforceArn"]
    cognito_config = wf_resp["Workforce"]["CognitoConfig"]
    user_pool_id = cognito_config["UserPool"]
    user_pool_client_id = cognito_config["ClientId"]
    print(f"ℹ️  Existing workforce found: {workforce_arn}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException":
        print("ℹ️  No existing workforce — will create one...")
    else:
        raise

# ── Step 2: Create Cognito User Pool (if no workforce exists) ──────────────────
cognito_client = session.client("cognito-idp")

if not workforce_arn:
    # Create a Cognito User Pool for the private workforce
    pool_resp = cognito_client.create_user_pool(
        PoolName="K21GroundTruthPool",
        AutoVerifiedAttributes=["email"],
        UsernameAttributes=["email"],
        Policies={
            "PasswordPolicy": {
                "MinimumLength": 8,
                "RequireUppercase": True,
                "RequireLowercase": True,
                "RequireNumbers": True,
                "RequireSymbols": False
            }
        }
    )
    user_pool_id = pool_resp["UserPool"]["Id"]
    print(f"✅ Cognito User Pool created: {user_pool_id}")

    # Create a User Pool Client (App Client)
    client_resp = cognito_client.create_user_pool_client(
        UserPoolId=user_pool_id,
        ClientName="K21GroundTruthClient",
        GenerateSecret=True,
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"]
    )
    user_pool_client_id = client_resp["UserPoolClient"]["ClientId"]
    print(f"✅ User Pool Client created: {user_pool_client_id}")

    # Create a Cognito Domain (required by SageMaker workforce)
    cognito_domain = f"k21-gt-{account_id}"
    try:
        cognito_client.create_user_pool_domain(
            Domain=cognito_domain,
            UserPoolId=user_pool_id
        )
        print(f"✅ Cognito domain created: {cognito_domain}")
    except ClientError as e:
        if "already exists" in str(e):
            print(f"ℹ️  Cognito domain already exists — reusing.")
        else:
            raise

    # Create the SageMaker Workforce (links to Cognito User Pool)
    wf_create_resp = sm_client.create_workforce(
        WorkforceName="default",
        CognitoConfig={
            "UserPool": user_pool_id,
            "ClientId": user_pool_client_id
        }
    )
    workforce_arn = wf_create_resp["WorkforceArn"]
    print(f"✅ SageMaker Workforce created: {workforce_arn}")

print(f"\n👥 Workforce ARN : {workforce_arn}")
print(f"   User Pool ID  : {user_pool_id}")

ℹ️  Existing workforce found: arn:aws:sagemaker:us-east-1:745404749079:workforce/default

👥 Workforce ARN : arn:aws:sagemaker:us-east-1:745404749079:workforce/default
   User Pool ID  : us-east-1_5ozXljgg7


---
## 👤 Cell 7: Create User Group, Invite Labeler & Build the Workteam

**Console equivalent:** Create Private Team → Enter email → Send invitation

The SageMaker API requires `CognitoMemberDefinition.UserGroup` to be a **non-empty Cognito Group name** (minimum length 1). The console hides this detail by auto-creating a group. In code we handle it explicitly in 4 steps:

1. **Create a Cognito User Group** (`K21LabelersGroup`)
2. **Create the labeler user** — triggers the invitation email with temp credentials
3. **Add the user to the group** so the workteam can filter by group membership
4. **Create the SageMaker workteam** referencing that group name

> ⚠️ Make sure `LABELER_EMAIL` in Cell 2 is your real email before running this cell.

In [8]:
# ── Step 1: Create a Cognito User Group ───────────────────────────────────────
# SageMaker requires CognitoMemberDefinition.UserGroup to be a non-empty string.
# We create a dedicated group for our labelers and add the user to it.
USER_GROUP_NAME = "K21LabelersGroup"

try:
    cognito_client.create_group(
        GroupName=USER_GROUP_NAME,
        UserPoolId=user_pool_id,
        Description="Labelers group for K21 Ground Truth private workforce"
    )
    print(f"✅ Cognito User Group '{USER_GROUP_NAME}' created.")
except ClientError as e:
    if e.response["Error"]["Code"] == "GroupExistsException":
        print(f"ℹ️  Group '{USER_GROUP_NAME}' already exists — reusing.")
    else:
        raise

# ── Step 2: Invite the labeler user ───────────────────────────────────────────
try:
    cognito_client.admin_create_user(
        UserPoolId=user_pool_id,
        Username=LABELER_EMAIL,
        UserAttributes=[
            {"Name": "email", "Value": LABELER_EMAIL},
            {"Name": "email_verified", "Value": "true"}
        ],
        DesiredDeliveryMediums=["EMAIL"]   # Sends invitation email with temp password
    )
    print(f"✅ Labeler invited! Check inbox: {LABELER_EMAIL}")
except ClientError as e:
    if e.response["Error"]["Code"] == "UsernameExistsException":
        print(f"ℹ️  User {LABELER_EMAIL} already exists in the pool.")
    else:
        raise

# ── Step 3: Add the user to the group ─────────────────────────────────────────
try:
    cognito_client.admin_add_user_to_group(
        UserPoolId=user_pool_id,
        Username=LABELER_EMAIL,
        GroupName=USER_GROUP_NAME
    )
    print(f"✅ User added to group '{USER_GROUP_NAME}'.")
except ClientError as e:
    print(f"⚠️  Could not add user to group: {e}")

# ── Step 4: Create the SageMaker private workteam ─────────────────────────────
# UserGroup must reference the Cognito group name created above (cannot be empty).
workteam_arn = None
try:
    wt_resp = sm_client.create_workteam(
        WorkteamName=TEAM_NAME,
        Description=f"Private labeling team for {ORG_NAME}",
        MemberDefinitions=[
            {
                "CognitoMemberDefinition": {
                    "UserPool": user_pool_id,
                    "UserGroup": USER_GROUP_NAME,   # Must be a valid, non-empty group name
                    "ClientId": user_pool_client_id
                }
            }
        ]
    )
    workteam_arn = wt_resp["WorkteamArn"]
    print(f"✅ Workteam '{TEAM_NAME}' created!")
    print(f"   Workteam ARN : {workteam_arn}")
except ClientError as e:
    error_msg = str(e).lower()
    if "already exists" in error_msg or "resourceinuse" in error_msg:
        # Fetch existing workteam ARN
        wt_list = sm_client.list_workteams(NameContains=TEAM_NAME)
        for wt in wt_list["Workteams"]:
            if wt["WorkteamName"] == TEAM_NAME:
                workteam_arn = wt["WorkteamArn"]
        print(f"ℹ️  Workteam already exists — ARN: {workteam_arn}")
    else:
        raise

print()
print("📧 Next step: Check your email for the labeling portal invitation!")
print("   You will use those credentials in Cell 10 to access the labeling UI.")

ℹ️  Group 'K21LabelersGroup' already exists — reusing.


ℹ️  User ritik@academy.com already exists in the pool.


✅ User added to group 'K21LabelersGroup'.


ℹ️  Workteam already exists — ARN: arn:aws:sagemaker:us-east-1:745404749079:workteam/private-crowd/K21Team

📧 Next step: Check your email for the labeling portal invitation!
   You will use those credentials in Cell 10 to access the labeling UI.


---
## 🔐 Cell 8: Configure the IAM Role for Ground Truth

**Console equivalent:** IAM Role → Create new role → Specify bucket name

SageMaker Ground Truth needs an IAM role that allows it to:
1. Read images from S3
2. Write labeled output back to S3
3. Access the labeling workforce (Cognito)

If your existing SageMaker role already has `AmazonSageMakerFullAccess` and `AmazonS3FullAccess`, you can reuse it directly.

In [9]:
iam_client = session.client("iam")

GT_ROLE_NAME = "SageMakerGroundTruthRole"

# Trust policy: allow SageMaker to assume this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

gt_role_arn = None
try:
    create_resp = iam_client.create_role(
        RoleName=GT_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="IAM role for SageMaker Ground Truth labeling jobs"
    )
    gt_role_arn = create_resp["Role"]["Arn"]
    print(f"✅ IAM Role created: {gt_role_arn}")

    # Attach AWS-managed policies
    for policy_arn in [
        "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
        "arn:aws:iam::aws:policy/AmazonS3FullAccess"
    ]:
        iam_client.attach_role_policy(RoleName=GT_ROLE_NAME, PolicyArn=policy_arn)
        print(f"   Attached: {policy_arn.split('/')[-1]}")

    print("⏳ Waiting 15 seconds for IAM role to propagate...")
    time.sleep(15)

except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        gt_role_arn = iam_client.get_role(RoleName=GT_ROLE_NAME)["Role"]["Arn"]
        print(f"ℹ️  IAM Role already exists — reusing: {gt_role_arn}")
    else:
        # Fall back to the notebook's own execution role
        gt_role_arn = role
        print(f"ℹ️  Using notebook execution role: {gt_role_arn}")

print(f"\n🔐 Ground Truth IAM Role ARN: {gt_role_arn}")

ℹ️  IAM Role already exists — reusing: arn:aws:iam::745404749079:role/SageMakerGroundTruthRole

🔐 Ground Truth IAM Role ARN: arn:aws:iam::745404749079:role/SageMakerGroundTruthRole


---
## 🏷️ Cell 9: Create the Image Classification Labeling Job

**Console equivalent:** Ground Truth → Labeling Jobs → Create Labeling Job

Two important fixes applied here compared to a naive implementation:

1. **`UiTemplateS3Uri` not `HumanTaskUiArn`** — `ImageMultiClassMultiLabel` requires a real HTML file in S3, not a built-in ARN.
2. **Labels hardcoded in the HTML template** — The Liquid variable `{{ task.input.labels }}` is unreliable for private workforce jobs and causes `FailedNonRetryableError`. Embedding the labels as a JSON array literal in the `categories` attribute of `<crowd-classifier-multi-select>` solves this.

> ℹ️ A new `JOB_NAME` is generated at the top of this cell so you can safely re-run it without hitting a "job already exists" error.

In [10]:
# ── Re-generate job name (needed if re-running after a failed job) ────────────
# Each labeling job name must be unique. Regenerate if running this cell again.
JOB_NAME = f"K21academy-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
print(f"🏷️  Job Name: {JOB_NAME}")

# ── Build the label category configuration ────────────────────────────────────
label_category_config = {
    "document-version": "2018-11-28",
    "labels": [{"label": lbl} for lbl in LABELS]
}
LABEL_CONFIG_KEY = f"{S3_INPUT_PREFIX}/label_categories.json"
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=LABEL_CONFIG_KEY,
    Body=json.dumps(label_category_config).encode("utf-8"),
    ContentType="application/json"
)
LABEL_CONFIG_S3_URI = f"s3://{BUCKET_NAME}/{LABEL_CONFIG_KEY}"
print(f"✅ Label config uploaded: {LABEL_CONFIG_S3_URI}")

# ── Build & upload the worker task UI template ────────────────────────────────
# WHY HARDCODE LABELS:
# The Liquid variable {{ task.input.labels }} is NOT reliably injected by Ground
# Truth for private workforce jobs — tasks silently fail with FailedNonRetryableError.
# Fix: embed the label list as a hardcoded JSON array directly in the HTML.
# The crowd-classifier-multi-select 'categories' attribute accepts a JSON array string.
labels_json = json.dumps(LABELS)   # e.g. '["Flight", "Building", "Happy Face"]'

UI_TEMPLATE_HTML = f"""<script src="https://assets.crowd.aws/crowd-html-elements.js"></script>
<crowd-form>
  <crowd-classifier-multi-select
    name="category"
    categories='{labels_json}'
    header="Select all labels that apply to this image"
  >
    <classification-target>
      <img style="max-width:100%" src="{{{{ task.input.taskObject | grant_read_access }}}}" />
    </classification-target>
    <full-instructions header="Labeling Instructions">
      <p>Look at the image carefully and select <strong>all</strong> labels that apply.</p>
      <ul>
        <li><b>Flight</b> — if there is an aircraft/plane visible</li>
        <li><b>Building</b> — if there is a building/structure visible</li>
        <li><b>Happy Face</b> — if there is a smiley or happy face visible</li>
      </ul>
    </full-instructions>
    <short-instructions>
      Select every label that applies. You may select more than one.
    </short-instructions>
  </crowd-classifier-multi-select>
</crowd-form>"""

UI_TEMPLATE_KEY = f"{S3_INPUT_PREFIX}/ui_template.html"
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=UI_TEMPLATE_KEY,
    Body=UI_TEMPLATE_HTML.encode("utf-8"),
    ContentType="text/html"
)
UI_TEMPLATE_S3_URI = f"s3://{BUCKET_NAME}/{UI_TEMPLATE_KEY}"
print(f"✅ UI template uploaded: {UI_TEMPLATE_S3_URI}")
print(f"   Labels embedded in template: {LABELS}")

# ── Create the labeling job ───────────────────────────────────────────────────
print(f"\n🚀 Creating labeling job: {JOB_NAME}")

create_job_resp = sm_client.create_labeling_job(
    LabelingJobName=JOB_NAME,
    LabelAttributeName="category",

    InputConfig={
        "DataSource": {
            "S3DataSource": {
                "ManifestS3Uri": MANIFEST_S3_URI
            }
        },
        "DataAttributes": {
            "ContentClassifiers": ["FreeOfPersonallyIdentifiableInformation"]
        }
    },

    OutputConfig={
        "S3OutputPath": OUTPUT_S3_URI
    },

    RoleArn=gt_role_arn,
    LabelCategoryConfigS3Uri=LABEL_CONFIG_S3_URI,

    HumanTaskConfig={
        "WorkteamArn": workteam_arn,

        # HTML template with labels hardcoded — avoids Liquid injection failures
        "UiConfig": {
            "UiTemplateS3Uri": UI_TEMPLATE_S3_URI
        },

        "PreHumanTaskLambdaArn": (
            f"arn:aws:lambda:{region}:432418664414:"
            "function:PRE-ImageMultiClassMultiLabel"
        ),

        "TaskTitle": "Image Classification (Multi-label): Label the images",
        "TaskDescription": "Select all applicable labels: Flight, Building, or Happy Face.",
        "TaskKeywords": ["image", "classification", "labeling"],
        "TaskTimeLimitInSeconds": 3600,
        "TaskAvailabilityLifetimeInSeconds": 2592000,
        "NumberOfHumanWorkersPerDataObject": 1,

        "AnnotationConsolidationConfig": {
            "AnnotationConsolidationLambdaArn": (
                f"arn:aws:lambda:{region}:432418664414:"
                "function:ACS-ImageMultiClassMultiLabel"
            )
        }
    }
)

LABELING_JOB_ARN = create_job_resp["LabelingJobArn"]
print(f"\n✅ Labeling job created successfully!")
print(f"   Job Name : {JOB_NAME}")
print(f"   Job ARN  : {LABELING_JOB_ARN}")
print(f"\n📧 Check your email ({LABELER_EMAIL}) for the labeling task portal link!")
print("   Open the portal, sign in, and label both images before running Cell 10.")

🏷️  Job Name: K21academy-20260612-085427
✅ Label config uploaded: s3://k21academyturk-745404749079/ground-truth/input/label_categories.json
✅ UI template uploaded: s3://k21academyturk-745404749079/ground-truth/input/ui_template.html
   Labels embedded in template: ['Flight', 'Building', 'Happy Face']

🚀 Creating labeling job: K21academy-20260612-085427



✅ Labeling job created successfully!
   Job Name : K21academy-20260612-085427
   Job ARN  : arn:aws:sagemaker:us-east-1:745404749079:labeling-job/K21academy-20260612-085427

📧 Check your email (ritik@academy.com) for the labeling task portal link!
   Open the portal, sign in, and label both images before running Cell 10.


---
## 📊 Cell 10: Monitor Labeling Job Status

**Console equivalent:** Ground Truth → Labeling Jobs → (view status column)

This cell polls the labeling job every 30 seconds and prints the current status.

> 🔑 **While this runs:** Open your email → click the portal link → sign in → complete the labeling tasks for both images → submit.
>
> Status flow: `Initializing` → `InProgress` → `Completed`

In [ ]:
print(f"🔄 Monitoring labeling job: {JOB_NAME}")
print("   ⏳ The task will appear in the portal within ~2 minutes.")
print("   ✍️  Complete labeling in the portal, then this cell will detect completion.")
print("   (Interrupt the kernel manually if you want to stop polling early.)\n")

terminal_statuses = {"Completed", "Failed", "Stopped"}
poll_interval     = 30   # seconds
max_polls         = 120  # poll up to 60 minutes total

for i in range(max_polls):
    job_info = sm_client.describe_labeling_job(LabelingJobName=JOB_NAME)
    status   = job_info["LabelingJobStatus"]
    stats    = job_info.get("LabelCounters", {})

    timestamp = datetime.now().strftime("%H:%M:%S")
    print(
        f"[{timestamp}] Status: {status:15s} | "
        f"Total: {stats.get('TotalLabeled',0)} | "
        f"HumanLabeled: {stats.get('HumanLabeled',0)} | "
        f"Unlabeled: {stats.get('Unlabeled',0)}"
    )

    if status in terminal_statuses:
        print(f"\n🎉 Job reached terminal state: {status}")
        break

    time.sleep(poll_interval)
else:
    print("\n⏱️  Polling timeout reached. Run the next cell to check final status.")

---
## 📂 Cell 11: Retrieve and Display the Labeled Output

**Console equivalent:** (No direct console equivalent — output is saved to S3 automatically)

Once the job is `Completed`, SageMaker writes an **output manifest** to S3. Each line in the output manifest contains the original image reference plus the human-assigned label(s). We read and display those results here.

In [ ]:
# Describe the completed job to get the output manifest location
job_info        = sm_client.describe_labeling_job(LabelingJobName=JOB_NAME)
current_status  = job_info["LabelingJobStatus"]
label_counters  = job_info.get("LabelCounters", {})

print(f"📋 Job Status   : {current_status}")
print(f"   Total Labeled: {label_counters.get('TotalLabeled', 0)}")
print(f"   Human Labeled: {label_counters.get('HumanLabeled', 0)}")
print()

if current_status == "Completed":
    output_config = job_info["OutputConfig"]
    output_s3_uri = output_config["S3OutputPath"]
    print(f"📦 Output saved to: {output_s3_uri}")

    # List files in the output S3 prefix
    output_prefix_parsed = output_s3_uri.replace(f"s3://{BUCKET_NAME}/", "")
    resp = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix=output_prefix_parsed
    )

    output_files = resp.get("Contents", [])
    print(f"\n📁 Output files ({len(output_files)} found):")
    for obj in output_files:
        print(f"   - s3://{BUCKET_NAME}/{obj['Key']}  ({obj['Size']} bytes)")

    # Read and parse the output manifest (file ending in 'output.manifest')
    for obj in output_files:
        if obj["Key"].endswith("output.manifest"):
            manifest_obj = s3_client.get_object(
                Bucket=BUCKET_NAME,
                Key=obj["Key"]
            )
            manifest_text = manifest_obj["Body"].read().decode("utf-8")

            print("\n📄 Labeled Output Manifest:")
            print("-" * 60)
            for line in manifest_text.strip().split("\n"):
                record = json.loads(line)
                image  = record.get("source-ref", "N/A").split("/")[-1]
                labels = record.get("category", {}).get("annotations", [])
                label_names = [LABELS[ann["label"]] for ann in labels] if labels else ["(unlabeled)"]
                print(f"   Image: {image:35s} → Labels: {', '.join(label_names)}")
            print("-" * 60)
            break
else:
    print(f"⏳ Job not yet complete (status: {current_status}).")
    print("   Please complete the labeling tasks in the portal and re-run this cell.")

---
## 🔍 Cell 12: (Optional) Quick Status Check — Run Anytime

Use this lightweight cell to check the job status without the polling loop.

In [ ]:
job_info = sm_client.describe_labeling_job(LabelingJobName=JOB_NAME)
print(f"Job Name   : {JOB_NAME}")
print(f"Status     : {job_info['LabelingJobStatus']}")
print(f"Created    : {job_info['CreationTime']}")
print(f"Label Stats: {job_info.get('LabelCounters', {})}")

---
## 🧹 Cell 13: Cleanup — Delete All Resources

**Console equivalent:** S3 → Empty bucket → Delete bucket | Ground Truth → Delete workforce

> ⚠️ **Run this only after completing the lab.** This will permanently delete:
> - All objects in the S3 bucket
> - The S3 bucket itself
> - The SageMaker workteam
> - The labeling job record
>
> **The IAM role and Cognito pool are left intact** (reusable for future labs).

In [12]:
# ── Safety prompt ──────────────────────────────────────────────────────────────
confirm = input(
    "⚠️  This will delete ALL lab resources. Type 'DELETE' to confirm: "
).strip()

if confirm != "DELETE":
    print("❌ Cleanup cancelled — no resources were deleted.")
else:
    print("🧹 Starting cleanup...\n")

    # 1. Empty and delete the S3 bucket
    try:
        # List and delete all objects
        paginator = s3_client.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=BUCKET_NAME):
            objects = page.get("Contents", [])
            if objects:
                s3_client.delete_objects(
                    Bucket=BUCKET_NAME,
                    Delete={"Objects": [{"Key": o["Key"]} for o in objects]}
                )
        print(f"✅ All objects deleted from '{BUCKET_NAME}'.")

        # Delete the bucket
        s3_client.delete_bucket(Bucket=BUCKET_NAME)
        print(f"✅ Bucket '{BUCKET_NAME}' deleted.")
    except ClientError as e:
        print(f"⚠️  S3 error: {e}")

    # 2. Delete the SageMaker workteam
    try:
        sm_client.delete_workteam(WorkteamName=TEAM_NAME)
        print(f"✅ Workteam '{TEAM_NAME}' deleted.")
    except ClientError as e:
        print(f"⚠️  Workteam error: {e}")

    # 3. Stop the labeling job if still running
    try:
        job_info = sm_client.describe_labeling_job(LabelingJobName=JOB_NAME)
        if job_info["LabelingJobStatus"] == "InProgress":
            sm_client.stop_labeling_job(LabelingJobName=JOB_NAME)
            print(f"✅ Labeling job '{JOB_NAME}' stopped.")
        else:
            print(f"ℹ️  Labeling job already in terminal state — no action needed.")
    except ClientError as e:
        print(f"⚠️  Labeling job error: {e}")

    print("\n🎉 Cleanup complete! No further charges will be incurred for these resources.")

---
## 📝 Lab Summary

Congratulations! 🎉 You have completed the **Image Classification Labeling Lab** using Amazon SageMaker Ground Truth — entirely through Python code.

### What You Accomplished:

| Step | Action | AWS Service |
|------|--------|-------------|
| 1 | Created an S3 bucket | Amazon S3 |
| 2 | Downloaded and uploaded images | Amazon S3 + `requests` |
| 3 | Generated an input manifest file | Amazon S3 |
| 4 | Created a Cognito User Pool + workforce | Amazon Cognito + SageMaker |
| 5 | Invited a labeler via email | Amazon Cognito |
| 6 | Created an IAM role for Ground Truth | AWS IAM |
| 7 | Launched an Image Classification labeling job | SageMaker Ground Truth |
| 8 | Monitored job progress programmatically | SageMaker Ground Truth |
| 9 | Retrieved and parsed labeled output | Amazon S3 |
| 10 | Cleaned up all resources | S3 + SageMaker |

### Key Concepts Reinforced:
- **Manifest files** are the bridge between your raw data and Ground Truth
- **Private workforces** use Amazon Cognito under the hood
- **Label Category Config** defines the classes shown to human labelers
- Ground Truth uses **Pre-annotation** and **Post-annotation Lambda ARNs** for built-in task types
- Labeled output is automatically written to S3 in JSON Lines format

---

### 📚 Further Reading
- [SageMaker Ground Truth Docs](https://docs.aws.amazon.com/sagemaker/latest/dg/sms-groundtruth.html)
- [Creating Labeling Jobs (API)](https://docs.aws.amazon.com/sagemaker/latest/dg/sms-labeling-job-create.html)
- [Built-in Task Types](https://docs.aws.amazon.com/sagemaker/latest/dg/sms-task-types.html)
- [Ground Truth Output Data Format](https://docs.aws.amazon.com/sagemaker/latest/dg/sms-data-output.html)